## Step 0 — Install the SDKs


%pip install --quiet --upgrade openai anthropic requests
print("Installed.")


In [1]:

%pip install --quiet --upgrade openai anthropic requests
print("Installed.")


Note: you may need to restart the kernel to use updated packages.
Installed.



## Step 1 — Load your API keys

Run this cell and paste each key when prompted. Nothing is written to disk;
the keys live only in this kernel's memory (`os.environ`) for the rest of
the session.


In [2]:

import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key: ")

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key: ")

print("Keys loaded into this session's environment.")


Keys loaded into this session's environment.


In [3]:
from openai import OpenAI
import anthropic

_openai_probe = OpenAI()
_anthropic_probe = anthropic.Anthropic()
print("A few OpenAI models your key can see:")

try:
    for m in list(_openai_probe.models.list())[:8]:
        print(" -", m.id)
except Exception as e:
    print("  (couldn't list models:", e, ")")

print("\nA few Anthropic models your key can see:")
try:
    for m in list(_anthropic_probe.models.list())[:8]:
        print(" -", m.id)
except Exception as e:
    print("  (couldn't list models:", e, ")")

A few OpenAI models your key can see:
 - tts-1
 - gpt-5-mini
 - gpt-5-mini-2025-08-07
 - gpt-5.4-2026-03-05
 - gpt-5-nano
 - gpt-5.4-mini
 - gpt-realtime-2.1-mini
 - gpt-realtime-2025-08-28

A few Anthropic models your key can see:
 - claude-haiku-5-5
 - claude-sonnet-5-5
 - claude-opus-5-5
 - claude-fable-5-1
 - claude-opus-5
 - claude-sonnet-5
 - claude-fable-5
 - claude-opus-4-8


In [4]:
OPENAI_MODEL = "gpt-5.4-mini"
ANTHROPIC_MODEL = "claude-sonnet-5-5"


PROMPT = "In one sentence, explain what a reverse proxy does."


## Part 1 — Feel the fragmentation

Same prompt, same intent, sent to both providers "directly" (no
HarnessRouter involved). Watch the *shape* of each call and each response,
not just the answer text.


In [5]:

# --- OpenAI, called directly -------------------------------------------------
from openai import OpenAI

openai_client = OpenAI()  # reads OPENAI_API_KEY from the environment

openai_resp = openai_client.responses.create(
    model=OPENAI_MODEL,
    input=PROMPT,
)

print("OpenAI answer:", openai_resp.output_text)
print()
print("Response object type:", type(openai_resp).__name__)
print("You read the answer via:  resp.output_text")
print("Conversation continuation via:  previous_response_id=resp.id")
print("Request field for the prompt:   input=...")


OpenAI answer: A reverse proxy sits in front of one or more servers, receives client requests, and forwards them to the appropriate backend server while hiding the servers’ details from the client.

Response object type: Response
You read the answer via:  resp.output_text
Conversation continuation via:  previous_response_id=resp.id
Request field for the prompt:   input=...


In [6]:

# --- Anthropic, called directly ----------------------------------------------
import anthropic

anthropic_client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment

anthropic_resp = anthropic_client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=1024,                         # <- OpenAI's call above needed no such field
    messages=[{"role": "user", "content": PROMPT}],
)

print("Anthropic answer:", anthropic_resp.content[0].text)
print()
print("Response object type:", type(anthropic_resp).__name__)
print("You read the answer via:  resp.content[0].text")
print("Conversation continuation via:  resend the full messages list yourself")
print("Request field for the prompt:   messages=[{'role': 'user', 'content': ...}]")


Anthropic answer: A reverse proxy sits in front of one or more backend servers, receiving client requests on their behalf and forwarding them to the appropriate server, which lets it handle tasks like load balancing, caching, SSL termination, and shielding the servers from direct exposure to the internet.

Response object type: Message
You read the answer via:  resp.content[0].text
Conversation continuation via:  resend the full messages list yourself
Request field for the prompt:   messages=[{'role': 'user', 'content': ...}]


In [7]:

# A function written "the OpenAI way" ...
def naive_ask(client, prompt):
    return client.responses.create(model=OPENAI_MODEL, input=prompt).output_text

print("Works against the OpenAI client:")
print(" ->", naive_ask(openai_client, PROMPT))

print()
print("Now the same function against the Anthropic client:")
try:
    naive_ask(anthropic_client, PROMPT)
except AttributeError as e:
    print(" -> AttributeError:", e)
    print("    (the Anthropic client simply has no `.responses` — there is no shared interface)")


Works against the OpenAI client:
 -> A reverse proxy sits in front of backend servers, receives client requests, and forwards them to the appropriate server while hiding and protecting the origin servers.

Now the same function against the Anthropic client:
 -> AttributeError: 'Anthropic' object has no attribute 'responses'
    (the Anthropic client simply has no `.responses` — there is no shared interface)



## Part 2 — Build your own miniature "unified gateway"

HarnessRouter's real API is a single endpoint:

```
POST /api/harness/v1/responses
{
  "input": "...",
  "metadata": {"harness_id": "codex"},
  "model": "gpt-5.4-mini",
  "stream": false
}
```

— OpenAI Responses-compatible, with the *harness* selected by a single
metadata field. Below, you'll implement a **toy version of exactly that
contract** yourself, in plain Python, backed by your own two keys. It won't
run real coding agents (that needs the actual Docker-based Runner — see
Part 3) but it demonstrates, with real model calls, the three things the
unified contract buys you:

1. **One call shape, many backends** — same function, `harness_id` picks
   the provider.
2. **Idempotency** — retrying a request with the same key doesn't re-run
   (or re-bill) the model call.
3. **Session continuation** — a `previous_response_id` continues a
   conversation the same way regardless of which provider is behind the
   harness (even though, as you saw above, the two providers don't agree on
   how to do this themselves).


In [8]:

import uuid
import time

HARNESS_BACKENDS = {
    "codex": "openai",
    "hermes": "openai",
    "claude": "anthropic",
    "claude-code": "anthropic",
}
_SESSIONS = {}      # response_id -> list of {"role", "content"} turns
_IDEMPOTENCY = {}   # idempotency_key -> the stored response dict

def call_harness(input_text, harness_id, model, previous_response_id=None,
                  idempotency_key=None):
    # A toy stand-in for POST /api/harness/v1/responses.
    #
    # Same call shape for every harness_id: `input`, `harness_id`, `model`,
    # optionally `previous_response_id` to continue a session and
    # `idempotency_key` to make retries safe.

    # --- idempotency: a repeated key returns the stored result, no model call ---
    if idempotency_key and idempotency_key in _IDEMPOTENCY:
        cached = dict(_IDEMPOTENCY[idempotency_key])
        cached["served_from_cache"] = True
        return cached

    backend = HARNESS_BACKENDS.get(harness_id)
    if backend is None:
        raise ValueError(f"unknown harness_id: {harness_id!r}")

    # --- build the turn history uniformly, regardless of backend ---
    history = list(_SESSIONS.get(previous_response_id, [])) if previous_response_id else []
    history.append({"role": "user", "content": input_text})

    # --- route to the right provider, normalize the result ---
    if backend == "openai":
        resp = openai_client.responses.create(model=model, input=history)
        output_text = resp.output_text
        usage = {"input_tokens": resp.usage.input_tokens,
                 "output_tokens": resp.usage.output_tokens}
    elif backend == "anthropic":
        resp = anthropic_client.messages.create(model=model, max_tokens=1024, messages=history)
        # Newer Claude models may return a ThinkingBlock first. Only TextBlocks have .text.
        output_text = "".join(
            block.text for block in resp.content if getattr(block, "type", None) == "text"
        )
        usage = {"input_tokens": resp.usage.input_tokens,
                 "output_tokens": resp.usage.output_tokens}
    else:
        raise ValueError(f"no backend wired up for: {backend!r}")

    response_id = f"resp_{uuid.uuid4().hex[:12]}"
    history.append({"role": "assistant", "content": output_text})
    _SESSIONS[response_id] = history

    result = {
        "id": response_id,
        "harness_id": harness_id,
        "backend": backend,
        "model": model,
        "output_text": output_text,
        "usage": usage,
        "served_from_cache": False,
    }
    if idempotency_key:
        _IDEMPOTENCY[idempotency_key] = result
    return result


print("Mini unified gateway ready: call_harness(input_text, harness_id, model, ...)")


Mini unified gateway ready: call_harness(input_text, harness_id, model, ...)


In [9]:

r1 = call_harness("Name one advantage of microservices, in a single sentence.",
                   harness_id="codex", model=OPENAI_MODEL)
print("[harness_id=codex -> backend:", r1["backend"], "]")
print(r1["output_text"])
print()

[harness_id=codex -> backend: openai ]
Microservices let teams develop, deploy, and scale parts of an application independently.



In [10]:
r2 = call_harness("Name one advantage of microservices, in a single sentence.",
                   harness_id="claude", model=ANTHROPIC_MODEL)
print("[harness_id=claude -> backend:", r2["backend"], "]")
print(r2["output_text"])

[harness_id=claude -> backend: anthropic ]
Microservices let teams develop, deploy, and scale each part of an application independently, which speeds up releases and allows resources to be allocated precisely where they're needed.


### 2b. Idempotency — retry safely without re-running (or re-billing) the task

In [11]:

key = str(uuid.uuid4())

t0 = time.time()
first = call_harness("Say exactly: it works.", harness_id="codex", model=OPENAI_MODEL,
                      idempotency_key=key)
t1 = time.time()
print(f"First call: {first['output_text']!r}  served_from_cache={first['served_from_cache']}  "
      f"({t1 - t0:.2f}s, a real model call)")

t2 = time.time()
second = call_harness("Say exactly: it works.", harness_id="codex", model=OPENAI_MODEL,
                       idempotency_key=key)
t3 = time.time()
print(f"Retry:      {second['output_text']!r}  served_from_cache={second['served_from_cache']}  "
      f"({t3 - t2:.4f}s, no model call at all)")

assert first["id"] == second["id"], "retry should return the SAME response id"
print("\nSame id both times:", first["id"] == second["id"],
      "-> the retry didn't start a second, separately-billed task.")


First call: 'it works.'  served_from_cache=False  (1.93s, a real model call)
Retry:      'it works.'  served_from_cache=True  (0.0001s, no model call at all)

Same id both times: True -> the retry didn't start a second, separately-billed task.



### 2c. Session continuation — works the same way whether the harness is backed
by OpenAI or Anthropic, even though (as Part 1 showed) the two providers
have no shared notion of "continue this conversation."


In [12]:

turn1 = call_harness("My favorite number is 42. Remember it.",
                      harness_id="claude", model=ANTHROPIC_MODEL)
print("Turn 1:", turn1["output_text"])

turn2 = call_harness("What's my favorite number?",
                      harness_id="claude", model=ANTHROPIC_MODEL,
                      previous_response_id=turn1["id"])
print("Turn 2:", turn2["output_text"])
print("\n(That worked via previous_response_id=turn1['id'] — the same parameter "
      "name HarnessRouter's real API uses, and it would work identically if "
      "turn1 had been backed by harness_id='codex' instead.)")


Turn 1: Got it, your favorite number is 42. 🌌

I'll remember it for this conversation. I can't promise it will carry over to future chats unless you're using a feature that saves memory across sessions. If that matters to you, you may want to note it somewhere yourself.

Is there a reason you like 42? The *Hitchhiker's Guide* is the obvious guess, but I'm curious.
Turn 2: Your favorite number is 42.

(That worked via previous_response_id=turn1['id'] — the same parameter name HarnessRouter's real API uses, and it would work identically if turn1 had been backed by harness_id='codex' instead.)



**On the machine where Docker runs, in the same folder as that file:**

```bash
docker compose up -d
docker compose logs -f harnessrouter   # wait for: [harnessrouter] ready on :3000
```

Then, once (per instance):

1. Open http://localhost:3000, sign in as `harnessrouter` with the
   password from `docker compose exec harnessrouter cat /data/selfhost-initial-password`.
2. Open **API Keys** in the Console, create one, and copy the secret —
   that's your `HARNESSROUTER_API_KEY`.

From here on, every call goes to **your own running instance** using the
real API, with a real coding-agent harness behind it instead of a chat
completion:


In [13]:
from dotenv import load_dotenv
load_dotenv()

True

In [14]:

import requests

HARNESSROUTER_BASE_URL = "http://localhost:3001/api/harness"
HARNESSROUTER_API_KEY = os.environ.get("HARNESSROUTER_API_KEY", "")

def call_real_harnessrouter(input_text, harness_id, model, stream=False):
    if not HARNESSROUTER_API_KEY:
        print("Set HARNESSROUTER_API_KEY (from the Console's API Keys page) "
              "and make sure `docker compose up -d` is running on this machine, "
              "then re-run this cell.")
        return None
    resp = requests.post(
        f"{HARNESSROUTER_BASE_URL}/v1/responses",
        headers={
            "Authorization": f"Bearer {HARNESSROUTER_API_KEY}",
            "content-type": "application/json",
        },
        json={
            "input": input_text,
            "metadata": {"harness_id": harness_id},
            "model": model,
            "stream": stream,
        },
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()

In [15]:
result = call_real_harnessrouter("Reply with exactly: it works.", harness_id="codex", model=OPENAI_MODEL)
print(result)


{'id': 'resp_df65137caa9949f4937dbb8cd80f79e1', 'object': 'response', 'created_at': 1794115736, 'status': 'completed', 'error': None, 'incomplete_details': None, 'previous_response_id': None, 'model': 'gpt-5.4-mini', 'output': [{'id': 'msg_5631a5a33709422bb8c96b4c3b2b5703', 'type': 'message', 'status': 'completed', 'role': 'assistant', 'content': [{'type': 'output_text', 'text': 'it works.', 'annotations': []}]}], 'store': True, 'usage': {'input_tokens': 11844, 'output_tokens': 17, 'total_tokens': 11861, 'output_tokens_details': {'reasoning_tokens': 8}}, 'connection': 'openai', 'served_model': 'gpt-5.4-mini-2026-03-17', 'reasoning': None, 'metadata': {'session_id': 'hsesse422dd327ee946619deb5dae291e938b'}}
